# Feature Selection & PCA

Feature Selection is the process of reducing the number of input variables when developing a predictive model. We generally divide these techniques into three categories:
1. **Filter Methods**: Using statistics (like correlation) to filter out bad features before the model even sees them.
2. **Embedded/Wrapper Methods**: Letting a machine learning model trial-and-error the features to tell us which ones it actually used.
3. **Feature Extraction (PCA)**: Mathematically squishing multiple features into a brand new, smaller set of features.

Let's set up a Python sandbox with a custom Real Estate dataset that intentionally contains redundant and useless data.

In [1]:
import pandas as pd
import numpy as np

# Create a dataset with 100 houses
np.random.seed(42)

# 1. Good Features
sqft = np.random.randint(1000, 5000, 100)
bedrooms = (sqft // 800) + np.random.randint(1, 3, 100)

# 2. Redundant Feature (Square Meters is just Sqft divided by 10.764)
sqft_meters = sqft / 10.764 

# 3. Useless Noise Feature (The shoe size of the person selling the house)
seller_shoe_size = np.random.randint(6, 14, 100)

# 4. The Target (Price depends on sqft and bedrooms, plus some random market noise)
price = (sqft * 200) + (bedrooms * 15000) + np.random.randint(-20000, 20000, 100)

# Build the DataFrame
df = pd.DataFrame({
    'sqft': sqft,
    'bedrooms': bedrooms,
    'sqft_meters': sqft_meters,
    'seller_shoe_size': seller_shoe_size,
    'price': price
})

print("✅ Dataset with Good, Redundant, and Useless features created!")
display(df.head())

✅ Dataset with Good, Redundant, and Useless features created!


,sqft,bedrooms,sqft_meters,seller_shoe_size,price
0,4174,7,387.774062,13,941356
1,4507,7,418.710517,12,1024959
2,1860,4,172.798216,6,438309
3,2294,4,213.117800,9,520719
4,2130,4,197.881828,10,470931


# 1. Filter Methods (Correlation Matrix)
The easiest way to find redundant features is to see if two columns are essentially telling you the exact same thing. We do this by checking their **Correlation**. 

Correlation ranges from `-1.0` to `1.0`. 
* If two features have a correlation near `1.0`, they move up together perfectly. 
* If they are near `-1.0`, one goes up while the other goes down. 
* If two input features are highly correlated with *each other* (usually $> 0.85$), you should drop one of them! Keeping both confuses linear models (a problem called Multicollinearity).

In [2]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the correlation matrix
corr_matrix = df.corr()

print("--- Correlation Matrix ---")
display(corr_matrix)

# We can see that 'sqft' and 'sqft_meters' have a correlation of exactly 1.0!
# The model doesn't need both. Let's drop the redundant one.
df_filtered = df.drop(columns=['sqft_meters'])

print("\n✅ Dropped 'sqft_meters' to prevent redundancy.")

--- Correlation Matrix ---


,sqft,bedrooms,sqft_meters,seller_shoe_size,price
sqft,1.000000,0.936199,1.000000,-0.017462,0.998252
bedrooms,0.936199,1.000000,0.936199,0.012152,0.947058
sqft_meters,1.000000,0.936199,1.000000,-0.017462,0.998252
seller_shoe_size,-0.017462,0.012152,-0.017462,1.000000,-0.016158
price,0.998252,0.947058,0.998252,-0.016158,1.000000



✅ Dropped 'sqft_meters' to prevent redundancy.


# 2. Embedded Methods (Tree Feature Importance)
Some Machine Learning algorithms, like **Random Forests** and **Decision Trees**, have feature selection built right into them. As they train, they keep track of which columns actually helped them reduce errors. 

We can train a quick Random Forest, ask it for its `.feature_importances_`, and use that to drop the useless noise.

In [3]:
from sklearn.ensemble import RandomForestRegressor

# Separate our inputs (X) from our target (y)
X = df_filtered.drop(columns=['price'])
y = df_filtered['price']

# Initialize and train a quick Random Forest model
model = RandomForestRegressor(random_state=42)
model.fit(X, y)

# Extract the feature importances
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("--- Random Forest Feature Importance ---")
display(importances)

# The model confirms 'seller_shoe_size' contributes almost 0% to predicting the price!
# Let's drop it.
X_final = X.drop(columns=['seller_shoe_size'])
print("\n✅ Dropped 'seller_shoe_size' because it lacks predictive power.")

--- Random Forest Feature Importance ---


,Feature,Importance
0,sqft,0.989794
1,bedrooms,0.009156
2,seller_shoe_size,0.001050



✅ Dropped 'seller_shoe_size' because it lacks predictive power.


# 3. Principal Component Analysis (PCA)
Filter and Embedded methods are **Feature Selection** (picking the best existing columns and throwing the rest in the trash). 

**PCA** is **Feature Extraction**. It does not throw columns away. Instead, it uses complex linear algebra to mathematically project your data into fewer dimensions, creating brand new columns (called Principal Components) that capture the maximum amount of variance (information) from the original data.

*Note: PCA is highly sensitive to the scale of your data. You **must** standardize your data (Lesson 04) before applying PCA!*

In [4]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Let's pretend we have a dataset with 10 features, and we want to squish it down to 2.
# We will use our original df (excluding the target price)
X_raw = df.drop(columns=['price'])

# 1. ALWAYS Standardize before PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# 2. Initialize PCA 
# We tell it: "Take my 4 columns and compress them down into 2 Principal Components"
pca = PCA(n_components=2)

# 3. Fit and Transform
X_pca = pca.fit_transform(X_scaled)

# Convert back to a DataFrame to view
df_pca = pd.DataFrame(X_pca, columns=['Principal_Component_1', 'Principal_Component_2'])

print("--- Data Compressed via PCA ---")
display(df_pca.head())

# How much of the original information did we keep?
explained_variance = pca.explained_variance_ratio_.sum() * 100
print(f"\n✅ By keeping only 2 components, we retained {explained_variance:.2f}% of the original data's information!")

--- Data Compressed via PCA ---


,Principal_Component_1,Principal_Component_2
0,1.886033,1.602028
1,2.237130,1.165871
2,-1.639307,-1.447686
3,-1.194744,-0.155902
4,-1.369193,0.278287



✅ By keeping only 2 components, we retained 97.90% of the original data's information!


*(Notice the output of PCA: We no longer have `sqft` or `bedrooms`. They are gone. We now have abstract mathematical components. This is the trade-off of PCA: You get incredible memory efficiency and speed, but you lose **Interpretability**. You can no longer explain to a boss *why* the model made a decision because the columns are no longer human-readable!)*

---

## Real-World Use Case or Analogy:
Think of Feature Selection and PCA like **Packing for a Backpacking Trip**:

* **Filter Method (Correlation)**: You are packing your bag and you notice you have a heavy Rain Jacket and a heavy Rain Poncho. They do the exact same thing (highly correlated). You throw the Poncho out to save weight.
* **Embedded Method (Importance)**: You pack a bowling ball. As you hike your first mile (training the model), you realize the bowling ball is completely useless for surviving in the wilderness (zero importance). You throw it out.
* **PCA (Feature Extraction)**: You have a sleeping bag, a heavy winter coat, and a giant foam mattress pad. They take up too much space. Instead of throwing them away, you mathematically compress them: you buy a high-tech, ultra-light space blanket that serves the function of all three items combined. You don't have your coat or mattress anymore (loss of interpretability), but you successfully retained the "Warmth and Sleep" information in a fraction of the space!

---